In [1]:
!pip install datasets sentence_transformers

In [2]:
# Create your API token from your Hugging Face Account. Make sure to save it in text file or notepad for future use.
# Will need to add it once per section
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from datasets import load_dataset
import time
import os
from typing import List, Optional

os.environ["OPENAI_API_KEY"] = "OPENAI_API_KEY"

class TextSimilarityModel:
    def __init__(self, corpus_name, rel_name, model_name='all-MiniLM-L6-v2', top_k=10):
        """
        Initialize the model with datasets and pre-trained sentence transformer.
        """
        self.model = SentenceTransformer(model_name)
        self.corpus_name = corpus_name
        self.rel_name = rel_name
        self.top_k = top_k
        self.load_data()


    def load_data(self):
        """
        Load and filter datasets based on test queries and documents.
        """
        # Load query and document datasets
        dataset_queries = load_dataset(self.corpus_name, "queries")
        dataset_docs = load_dataset(self.corpus_name, "corpus")

        # Extract queries and documents
        self.queries = dataset_queries["queries"]["text"]
        self.query_ids = dataset_queries["queries"]["_id"]
        self.documents = dataset_docs["corpus"]["text"]
        self.document_ids = dataset_docs["corpus"]["_id"]


        # Filter queries and documents and build relevant queries and documents mapping based on test set
        test_qrels = load_dataset(self.rel_name)["test"]
        self.filtered_test_query_ids = set(test_qrels["query-id"])
        self.filtered_test_doc_ids = set(test_qrels["corpus-id"])

        self.test_queries = [q for qid, q in zip(self.query_ids, self.queries) if qid in self.filtered_test_query_ids]
        self.test_query_ids = [qid for qid in self.query_ids if qid in self.filtered_test_query_ids]
        self.test_documents = [doc for did, doc in zip(self.document_ids, self.documents) if did in self.filtered_test_doc_ids]
        self.test_document_ids = [did for did in self.document_ids if did in self.filtered_test_doc_ids]

        self.test_query_id_to_relevant_doc_ids = {qid: [] for qid in self.test_query_ids}
        for qid, doc_id in zip(test_qrels["query-id"], test_qrels["corpus-id"]):
            if qid in self.test_query_id_to_relevant_doc_ids:
                self.test_query_id_to_relevant_doc_ids[qid].append(doc_id)

        ## Code Below this is used for creating the training set
        # Build query and document id to text mapping
        self.query_id_to_text = {query_id:query for query_id, query in zip(self.query_ids, self.queries)}
        self.document_id_to_text = {document_id:document for document_id, document in zip(self.document_ids, self.documents)}

        # Build relevant queries and documents mapping based on train set
        train_qrels = load_dataset(self.rel_name)["train"]
        self.train_query_id_to_relevant_doc_ids = {qid: [] for qid in train_qrels["query-id"]}

        for qid, doc_id in zip(train_qrels["query-id"], train_qrels["corpus-id"]):
            if qid in self.train_query_id_to_relevant_doc_ids:
                # Append the document ID to the relevant doc mapping
                self.train_query_id_to_relevant_doc_ids[qid].append(doc_id)

        # Filter queries and documents and build relevant queries and documents mapping based on validation set
        #TODO Put your code here.
        ###########################################################################

        val_qrels = load_dataset(self.rel_name)["validation"]
        self.filtered_val_query_ids = set(val_qrels["query-id"])
        self.filtered_val_doc_ids = set(val_qrels["corpus-id"])

        self.val_queries = [q for qid, q in zip(self.query_ids, self.queries) if qid in self.filtered_val_query_ids]
        self.val_query_ids = [qid for qid in self.query_ids if qid in self.filtered_val_query_ids]
        self.val_documents = [doc for did, doc in zip(self.document_ids, self.documents) if did in self.filtered_val_doc_ids]
        self.val_document_ids = [did for did in self.document_ids if did in self.filtered_val_doc_ids]

        self.val_query_id_to_relevant_doc_ids = {qid: [] for qid in self.val_query_ids}
        for qid, doc_id in zip(val_qrels["query-id"], val_qrels["corpus-id"]):
            if qid in self.val_query_id_to_relevant_doc_ids:
                self.val_query_id_to_relevant_doc_ids[qid].append(doc_id)
        ###########################################################################


    #Task 1: Encode Queries and Documents (10 Pts)

    def encode_with_glove(self, glove_file_path: str, sentences: list[str]) -> list[np.ndarray]:

        """
        # Inputs:
            - glove_file_path (str): Path to the GloVe embeddings file (e.g., "glove.6B.50d.txt").
            - sentences (list[str]): A list of sentences to encode.

        # Output:
            - list[np.ndarray]: A list of sentence embeddings

        (1) Encodes sentences by averaging GloVe 50d vectors of words in each sentence.
        (2) Return a sequence of embeddings of the sentences.
        Download the glove vectors from here.
        https://nlp.stanford.edu/data/glove.6B.zip
        Handle unknown words by using zero vectors
        """
        #TODO Put your code here.
        ###########################################################################

        # Load GloVe embeddings into a dictionary
        glove = {}
        # Determine the GloVe vector dimension (from file name or by parsing first line)
        with open(glove_file_path, 'r', encoding='utf8') as f:
            for line in f:
                splits = line.strip().split()
                word = splits[0]
                vec = np.array([float(x) for x in splits[1:]], dtype=np.float32)
                glove[word] = vec
        vec_dim = next(iter(glove.values())).shape[0] if glove else 50  # fallback to 50 if glove empty

        embeddings = []
        for sent in sentences:
            words = sent.strip().split()
            vecs = []
            for w in words:
                if w in glove:
                    vecs.append(glove[w])
                else:
                    vecs.append(np.zeros(vec_dim, dtype=np.float32))
            if len(vecs) == 0:
                # Handle completely empty sentence
                sent_vec = np.zeros(vec_dim, dtype=np.float32)
            else:
                sent_vec = np.mean(vecs, axis=0)
            embeddings.append(sent_vec)
        return embeddings
        ###########################################################################


    def encode_with_openai(
        self,
        sentences: List[str],
        model: str = 'text-embedding-3-small',
        api_key: Optional[str] = None,
        batch_size: int = 100
    ) -> np.ndarray:
        """
        Encodes sentences using OpenAI's embedding API.

        # Inputs:
            - sentences (List[str]): A list of sentences to encode.
            - model (str): OpenAI model name. Options:
                * 'text-embedding-3-small' (1536 dims, $0.02/1M tokens) - RECOMMENDED
                * 'text-embedding-3-large' (3072 dims, $0.13/1M tokens)
                * 'text-embedding-ada-002' (1536 dims, legacy)
            - api_key (str, optional): OpenAI API key. If None, reads from OPENAI_API_KEY env variable
            - batch_size (int): Number of sentences to encode per API call (max 2048)

        Instructions:
        - Implement batched encoding with error handling
        - Add rate limiting (sleep between batches)

        Expected Cost for this Assignment:
        - ~4,000 texts (320 queries + 3,600 documents)
        - text-embedding-3-small: ~$0.08-0.10 per student
        - text-embedding-3-large: ~$0.50-0.65 per student

        Tips:
        - Use try-except for API errors
        - Implement retry logic with exponential backoff
        - Cache embeddings to avoid re-encoding
        - Monitor your usage at: https://platform.openai.com/usage
        """
        #TODO Put your code here.
        ###########################################################################

        def _openai_client(api_key):
            try:
                from openai import OpenAI
            except ImportError:
                raise ImportError("You must install the openai package (pip install openai)")
            return OpenAI(api_key=api_key)

        def _get_openai_api_key(user_api_key):
            if user_api_key is not None:
                return user_api_key
            env_key = os.environ.get("OPENAI_API_KEY", None)
            if env_key is None:
                raise ValueError("OpenAI API key not found. Set OPENAI_API_KEY env var or pass api_key")
            return env_key

        # get the openai api key
        final_api_key = _get_openai_api_key(api_key)
        client = _openai_client(final_api_key)

        all_embeddings = []
        n = len(sentences)

        # Rate limiting settings
        sleep_sec = 1.3
        max_retries = 5

        for start in range(0, n, batch_size):
            batch = sentences[start : start + batch_size]

            # Retry mechanism
            for attempt in range(max_retries):
                try:
                    response = client.embeddings.create(
                        input=batch,
                        model=model
                    )
                    # Extract embeddings
                    batch_embs = [x.embedding for x in response.data]
                    all_embeddings.extend(batch_embs)
                    break  # Success!

                except Exception as e:
                    if attempt < max_retries - 1:
                        backoff = sleep_sec * (2 ** attempt)
                        print(f"OpenAI API error: {e}. Retrying in {backoff:.1f}s...")
                        time.sleep(backoff)
                    else:
                        print(f"Failed after {max_retries} attempts. Raising exception.")
                        raise e

            # Rate limiting between batches
            if start + batch_size < n:
                time.sleep(sleep_sec)

        return np.array(all_embeddings)

        ###########################################################################

    #Task 2: Calculate Cosine Similarity and Rank Documents (20 Pts)

    def rank_documents(self, encoding_method: str = 'sentence_transformer') -> None:
        """
         # Inputs:
            - encoding_method (str): The method used for encoding queries/documents.
                             Options: ['glove', 'sentence_transformer'].

        # Output:
            - None (updates self.query_id_to_ranked_doc_ids with ranked document IDs).

        (1) Compute cosine similarity between each document and the query
        (2) Rank documents for each query and save the results in a dictionary "query_id_to_ranked_doc_ids"
            This will be used in "mean_average_precision"
            Example format {2: [125, 673], 35: [900, 822]}
        """
        # get the target documents and document ids
        target_docs = self.documents
        target_doc_ids = self.document_ids

        if encoding_method == 'glove':
            # Note: Ensure "glove.6B.50d.txt" is downloaded and in the local directory
            query_embeddings = self.encode_with_glove("glove.6B.50d.txt", self.test_queries)
            document_embeddings = self.encode_with_glove("glove.6B.50d.txt", target_docs)
        elif encoding_method == 'sentence_transformer':
            query_embeddings = self.model.encode(self.test_queries)
            document_embeddings = self.model.encode(target_docs)
        elif encoding_method == 'openai':
            # Use environment variable or prompt for API key
            if hasattr(self, 'cached_encoding_method') and self.cached_encoding_method == 'openai' and hasattr(self, 'cached_doc_embeddings'):
                print("Using cached OpenAI embeddings for documents...")
                document_embeddings = self.cached_doc_embeddings
            else:
                print("Encoding documents with OpenAI API (this may cost money)...")
                document_embeddings = self.encode_with_openai(target_docs)
            # encode the queries
            query_embeddings = self.encode_with_openai(self.test_queries)
        else:
            raise ValueError("Invalid encoding method.")

        # cache the document embeddings and document ids
        self.cached_doc_embeddings = document_embeddings
        self.cached_doc_ids = target_doc_ids
        self.cached_encoding_method = encoding_method

        #TODO Put your code here.
        ###########################################################################
         # define a dictionary to store the ranked documents for each query
        self.query_id_to_ranked_doc_ids = {}

        # compute the cosine similarity between the query embeddings and the document embeddings
        similarity_matrix = cosine_similarity(query_embeddings, document_embeddings)

        # rank the documents for each query
        for i, qid in enumerate(self.test_query_ids):
            scores = similarity_matrix[i]
            doc_score_pairs = list(zip(target_doc_ids, scores))

            sorted_docs = sorted(doc_score_pairs, key=lambda x: x[1], reverse=True)
            top_k_doc_ids = [doc_id for doc_id, score in sorted_docs[:self.top_k]]
            # store the ranked documents for each query
            self.query_id_to_ranked_doc_ids[qid] = top_k_doc_ids
        ###########################################################################

    @staticmethod
    def average_precision(relevant_docs: list[str], candidate_docs: list[str]) -> float:
        """
        # Inputs:
            - relevant_docs (list[str]): A list of document IDs that are relevant to the query.
            - candidate_docs (list[str]): A list of document IDs ranked by the model.

        # Output:
            - float: The average precision score

        Compute average precision for a single query.
        """
        y_true = [1 if doc_id in relevant_docs else 0 for doc_id in candidate_docs]
        precisions = [np.mean(y_true[:k+1]) for k in range(len(y_true)) if y_true[k]]
        return np.mean(precisions) if precisions else 0

    #Task 3: Calculate Evaluate System Performance (10 Pts)

    def mean_average_precision(self) -> float:
        """
        # Inputs:
            - None (uses ranked documents stored in self.query_id_to_ranked_doc_ids).

        # Output:
            - float: The MAP score, computed as the mean of all average precision scores.

        (1) Compute mean average precision for all queries using the "average_precision" function.
        (2) Compute the mean of all average precision scores
        Return the mean average precision score

        reference: https://www.evidentlyai.com/ranking-metrics/mean-average-precision-map
        https://towardsdatascience.com/map-mean-average-precision-might-confuse-you-5956f1bfa9e2
        """
         #TODO Put your code here.
        ###########################################################################

        average_precisions = []
        for query_id, ranked_doc_ids in self.query_id_to_ranked_doc_ids.items():
            # Get relevant doc ids for this query from self.qrels
            relevant_docs = self.test_query_id_to_relevant_doc_ids.get(query_id, [])

            # if there are no relevant documents, skip the query
            if not relevant_docs:
                continue

            ap = self.average_precision(relevant_docs, ranked_doc_ids)
            average_precisions.append(ap)
        return np.mean(average_precisions) if average_precisions else 0.0
        ###########################################################################

    #Task 4: Ranking the Top 10 Documents based on Similarity Scores (10 Pts)

    def show_ranking_documents(self, encoding_method: str, example_query: str) -> None:

        """
        # Inputs:
            - example_query (str): A query string for which top-ranked documents should be displayed.

        # Output:
            - None (prints the ranked documents along with similarity scores).

        (1) rank documents with given query with cosine similarity scores
        (2) prints the top 10 results along with its similarity score.

        """
        #TODO Put your code here.

        ###########################################################################

        # 1. Encode the single query based on the method
        # 2. Reshape check: Ensure query_embedding is (1, n_features)
        # 3. Calculate scores

        # check if the document embeddings are cached
        if hasattr(self, 'cached_doc_embeddings') and \
           hasattr(self, 'cached_encoding_method') and \
           self.cached_encoding_method == encoding_method:

            # if the document embeddings are cached, use the cached document embeddings and document ids
            doc_embeddings = self.cached_doc_embeddings
            doc_ids_list = self.cached_doc_ids

        else:
            # if the document embeddings are not cached, encode the documents
            print(f"Warning: Re-encoding ALL documents with {encoding_method}. Run rank_documents() first to save time/money!")
            if encoding_method == "glove":
                doc_embeddings = self.encode_with_glove("glove.6B.50d.txt", self.documents)
            elif encoding_method == "sentence_transformer":
                doc_embeddings = self.model.encode(self.documents)
            elif encoding_method == "openai":
                doc_embeddings = self.encode_with_openai(self.documents)
            doc_ids_list = self.document_ids

        # encode the query
        if encoding_method == "glove":
            query_embedding = self.encode_with_glove("glove.6B.50d.txt", [example_query])
        elif encoding_method == "sentence_transformer":
            query_embedding = self.model.encode([example_query])
        elif encoding_method == "openai":
            query_embedding = self.encode_with_openai([example_query])

        # compute the cosine similarity between the query embedding and the document embeddings
        similarity_scores = cosine_similarity(query_embedding, doc_embeddings)[0]

        # create a list of tuples containing the document id, the document text, and the similarity score
        doc_data = []
        for i, doc_id in enumerate(doc_ids_list):
             doc_data.append((doc_id, self.document_id_to_text[doc_id], similarity_scores[i]))

        ranked_docs = sorted(doc_data, key=lambda x: x[2], reverse=True)

        print(f"Top 10 ranked documents for query: '{example_query}' (method: {encoding_method})")
        print("-" * 80)
        for idx, (doc_id, text, score) in enumerate(ranked_docs[:10], 1):
            print(f"{idx}. Doc ID: {doc_id}\n   Score: {score:.4f}\n   Text: {text[:200].strip()}...\n")
        ###########################################################################

    #Task 5:Fine tune the sentence transformer model (25 Pts)
    # Students are not graded on achieving a high MAP score.
    # The key is to show understanding, experimentation, and thoughtful analysis.

    def fine_tune_model(self, batch_size: int = 32, num_epochs: int = 3, save_model_path: str = "finetuned_senBERT") -> None:

        """
        Fine-tunes the model using MultipleNegativesRankingLoss.
        (1) Prepare training examples from `self.prepare_training_examples()`
        (2) Experiment with [anchor, positive] vs [anchor, positive, negative]
        (3) Define a loss function (`MultipleNegativesRankingLoss`)
        (4) Freeze all model layers except the final layers
        (5) Train the model with the specified learning rate
        (6) Save the fine-tuned model
        """
        #TODO Put your code here.
        ###########################################################################
        # import DataLoader
        from torch.utils.data import DataLoader

        # Prepare training examples
        train_examples = self.prepare_training_examples()
        train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=batch_size)

        # Define loss function
        train_loss = losses.MultipleNegativesRankingLoss(model=self.model)

        # Freeze all model layers except the final layers
        for param in self.model.parameters():
            param.requires_grad = False

        # Unfreeze the final layers
        print("Freezing early layers... Only training the final layer.")
        for name, param in self.model.named_parameters():
            if 'layer.5' in name or 'pooler' in name:
                param.requires_grad = True

        # Calculate warmup steps
        warmup_steps = int(len(train_dataloader) * num_epochs * 0.1)

        self.model.fit(
            train_objectives=[(train_dataloader, train_loss)],
            epochs=num_epochs,
            warmup_steps=warmup_steps,
            output_path=save_model_path,
            show_progress_bar=True
        )

        # Save the fine-tuned model
        print(f"Fine-tuning complete. Model saved to {save_model_path}")
        ###########################################################################

    # Take a careful look into how the training set is created
    def prepare_training_examples(self) -> list[InputExample]:
        """
        Prepares training examples from the training data.
        # Inputs:
            - None (uses self.train_query_id_to_relevant_doc_ids to create training pairs).

         # Output:
            Output: - list[InputExample]: A list of training samples containing [anchor, positive] or [anchor, positive, negative].

        """
        # import random
        import random

        train_examples = []
        # create a list of all document ids
        all_doc_ids_list = self.document_ids

        for qid, doc_ids in self.train_query_id_to_relevant_doc_ids.items():
            # create a set of relevant documents
            relevant_set = set(doc_ids)

            for doc_id in doc_ids:
                anchor = self.query_id_to_text[qid]
                positive = self.document_id_to_text[doc_id]
                # TODO: Select random negative examples that are not relevant to the query.
                # TODO: Create list[InputExample] of type [anchor, positive, negative]

                # select a random negative example that is not in the relevant set
                while True:
                    neg_doc_id = random.choice(all_doc_ids_list)
                    if neg_doc_id not in relevant_set:
                        break

                negative = self.document_id_to_text[neg_doc_id]
                # create a list of negative examples
                train_examples.append(InputExample(texts=[anchor, positive, negative]))

        return train_examples


In [4]:
# Initialize the model with the medical dataset (nfcorpus)
model = TextSimilarityModel("BeIR/nfcorpus", "BeIR/nfcorpus-qrels")

# Evaluate using the default Sentence Transformer
print("Ranking with sentence_transformer...")
model.rank_documents(encoding_method='sentence_transformer')
sbert_map = model.mean_average_precision()
print("SBERT Mean Average Precision:", sbert_map)

# Evaluate using GloVe (requires 'glove.6B.50d.txt' in your directory)
print("\nRanking with glove...")
model.rank_documents(encoding_method='glove')
glove_map = model.mean_average_precision()
print("GloVe Mean Average Precision:", glove_map)

# Qualitative test: Show actual document text for a sample query
model.show_ranking_documents("glove","Breast Cancer Cells Feed on Cholesterol")
model.show_ranking_documents("sentence_transformer", "Breast Cancer Cells Feed on Cholesterol")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

test.tsv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/110575 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11385 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12334 [00:00<?, ? examples/s]

Ranking with sentence_transformer...
SBERT Mean Average Precision: 0.4587536535081868

Ranking with glove...
GloVe Mean Average Precision: 0.054143323996265175
Top 10 ranked documents for query: 'Breast Cancer Cells Feed on Cholesterol' (method: glove)
--------------------------------------------------------------------------------
1. Doc ID: MED-1515
   Score: 0.9177
   Text: Long periods of sedentary behaviour may adversely affect health irrespective of overall physical activity levels. This study compared the effects of sitting, standing and walking on postprandial lipae...

2. Doc ID: MED-3659
   Score: 0.9150
   Text: We report the annual results of patch testing with lavender oil for a 9-year period from 1990 to 1998 in Japan. Using Finn Chambers and Scanpor tape, we performed 2-day closed patch testing with laven...

3. Doc ID: MED-1524
   Score: 0.9068
   Text: Ever since smoking was prohibited in restaurants, bars, and clubs, undesirable smells that were previously masked by c

In [5]:
print("\nRanking with openai...")
model.rank_documents(encoding_method='openai')
openai_map = model.mean_average_precision()
print("openai Mean Average Precision:", openai_map)

model.show_ranking_documents("openai","Breast Cancer Cells Feed on Cholesterol")


Ranking with openai...
Encoding documents with OpenAI API (this may cost money)...
openai Mean Average Precision: 0.5258482213548187
Top 10 ranked documents for query: 'Breast Cancer Cells Feed on Cholesterol' (method: openai)
--------------------------------------------------------------------------------
1. Doc ID: MED-2434
   Score: 0.5738
   Text: The specific role of dietary fat in breast cancer progression is unclear, although a low-fat diet was associated with decreased recurrence of estrogen receptor alpha negative (ER(-)) breast cancer. ER...

2. Doc ID: MED-2439
   Score: 0.5047
   Text: While many factors are involved in the etiology of cancer, it has been clearly established that diet significantly impacts one’s risk for this disease. More recently, specific food components have bee...

3. Doc ID: MED-2427
   Score: 0.4960
   Text: Lipid rafts/caveolae are membrane platforms for signaling molecules that regulate various cellular functions, including cell survival. To bette

In [6]:
# Finetune all-MiniLM-L6-v2 sentence transformer model
model.fine_tune_model(batch_size=3, num_epochs=2, save_model_path="finetuned_senBERT_train_v2")  # Adjust batch size and epochs as needed

model.rank_documents()
map_score = model.mean_average_precision()
print("Mean Average Precision:", map_score)

Freezing early layers... Only training the final layer.


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,2.068056
1000,2.004257
1500,1.876245
2000,1.744521
2500,1.714557
3000,1.603763
3500,1.534477
4000,1.542471
4500,1.511601
5000,1.490381


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuning complete. Model saved to finetuned_senBERT_train_v2
Mean Average Precision: 0.44845937839073247
